# 03 · Linear Regression
> **Part:** Part 1 · Classical supervised learning  
> **Family:** `Supervised · Regression`

## Definition
Linear regression models a numeric target as a weighted sum of the input features plus a bias:
ŷ = w·x + b. Training finds the weights that minimize the mean squared error between predictions and truth.

## Intuition
Draw the straight line that passes as close as possible to all the points, where "close" means the *sum of squared
vertical distances (residuals)* is as small as it can be. Squaring punishes big misses and makes the math smooth.


## The Math

```text
Loss      L(w,b) = 1/(2n) Σ (w·xᵢ + b − yᵢ)²
    Gradient  ∂L/∂w = 1/n Σ (ŷᵢ − yᵢ)·xᵢ        ∂L/∂b = 1/n Σ (ŷᵢ − yᵢ)
    Update    w ← w − η ∂L/∂w ,  b ← b − η ∂L/∂b
    Closed form (normal equation): w = (XᵀX)⁻¹ Xᵀ y
```


In [ ]:
# Setup, Imports & Styling
%matplotlib inline
import os
import pathlib
import sys
from IPython.display import Image, display

# Ensure parent folder is in path to import mlatlas
parent_dir = pathlib.Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlatlas import data, viz, cli

# Ensure interactive inline mode and configure Seaborn theme
plt.ion()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})


### Step 1: Exploratory Data Visualization with Seaborn
We examine synthetic weekly advertising spend versus units sold. The relationship shows a clear linear trend with Gaussian noise, alongside a multiple regression dataset of house prices.


In [ ]:
X_ads, y_ads = data.sales_vs_ads()
x_ads = X_ads[:, 0]

X_house, y_house, house_feat_names = data.house_prices(n=300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Simple Linear Regression Data (Ad spend vs Sales)
sns.scatterplot(x=x_ads, y=y_ads, color="#3b6fb6", s=60, alpha=0.85, edgecolor="w", ax=ax1)
ax1.set_title("Univariate Dataset: Ad Spend vs. Units Sold", fontsize=11, fontweight="bold")
ax1.set_xlabel("Ad Spend ($1,000s)")
ax1.set_ylabel("Units Sold")

# 2. House Prices correlation heatmap
df_house = pd.DataFrame(X_house, columns=house_feat_names)
df_house["price_k"] = y_house
sns.heatmap(df_house.corr(), annot=True, fmt=".2f", cmap="vlag", ax=ax2)
ax2.set_title("Multivariate Dataset: House Prices Correlation Matrix", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()


### Step 2: From-Scratch Algorithm Implementation
Below is the complete implementation of the model components in pure NumPy.


In [ ]:
# Main Algorithm & Training Routine
def main(gif, quick):
    X, y = data.sales_vs_ads(); x = X[:, 0]; n = len(y)
    steps = 800 if quick else 2500; lr = .03; w, b = -1.0, 12.0; hist = [(w, b)]
    for _ in range(steps):
        err = w * x + b - y; w, b = w - lr * (err * x).mean(), b - lr * err.mean(); hist.append((w, b))
    wb = np.linalg.lstsq(np.c_[x, np.ones(n)], y, rcond=None)[0]
    print(f"  gradient descent: w={w:.3f} b={b:.3f}   normal equation: w={wb[0]:.3f} b={wb[1]:.3f}   (truth: w=2.2, b=3.0)")
    # --- part B: multiple regression on house prices
    Xh, yh, names = data.house_prices(300); Zh, mu, sd = data.standardize(Xh); ym = yh.mean()
    wh = np.zeros(4); bh = 0.
    for _ in range(2000):
        e = Zh @ wh + bh - (yh - ym); wh -= .1 * Zh.T @ e / len(e); bh -= .1 * e.mean()
    ne = np.linalg.lstsq(np.c_[Zh, np.ones(len(yh))], yh - ym, rcond=None)[0]
    r2 = 1 - ((Zh @ wh + bh - (yh - ym)) ** 2).sum() / ((yh - ym) ** 2).sum()
    print("  house prices (standardized features):   feature      GD     normal-eq")
    for nm, a, c in zip(names, wh, ne[:4]): print(f"                                          {nm:10s} {a:7.2f} {c:7.2f}")
    print(f"  R² = {r2:.3f}")
    Wg, Bg = np.meshgrid(np.linspace(-1, 5, 70), np.linspace(-6, 14, 70))
    L = (((Wg[..., None] * x + Bg[..., None] - y) ** 2).mean(-1)) / 2
    idx = viz.geo(len(hist), 14 if quick else 60); hp = np.array(hist)
    fig, (a1, a2) = viz.subplots(1, 2, w=11, h=4.4)

    def update(i):
        a1.clear(); a2.clear(); w_, b_ = hist[i]; xs = np.array([0, 10])
        a1.scatter(x, y, s=16, color=viz.PALETTE[0], zorder=3); a1.plot(xs, w_ * xs + b_, color=viz.PALETTE[3], lw=2.4)
        a1.vlines(x, y, w_ * x + b_, color=viz.PALETTE[3], alpha=.35)
        a1.set(title=f"iteration {i}:  y = {w_:.2f}·x + {b_:.2f}   MSE = {np.mean((w_ * x + b_ - y) ** 2):.1f}", xlabel="ad spend (k$)", ylabel="units sold", ylim=(-5, 35))
        a2.contourf(Wg, Bg, L, levels=np.geomspace(L.min() + .5, L.max(), 18), cmap="Blues_r", alpha=.8)
        a2.plot(hp[:i + 1, 0], hp[:i + 1, 1], color="k", lw=1.2); a2.scatter(w_, b_, s=50, color=viz.PALETTE[3], zorder=5, edgecolor="k")
        a2.scatter(*wb, marker="*", s=140, color="gold", edgecolor="k", zorder=6)
        a2.set(title="Loss surface L(w, b), gradient descent path", xlabel="slope w", ylabel="intercept b"); a2.grid(False)
    viz.animate(fig, update, idx, gif, fps=8)


### Step 3: Fitted Model, Residual Diagnostics & Loss Surface
We visualize: (1) The fitted univariate regression line with residual error segments, (2) The 2D loss surface $L(w, b)$ with the gradient descent trajectory path, and (3) Multiple regression feature weights.


In [ ]:
# 1. Fit Univariate Model with Gradient Descent
w, b = -1.0, 12.0
lr = 0.03
n = len(y_ads)
hist = [(w, b)]
for _ in range(1200):
    err = w * x_ads + b - y_ads
    w -= lr * (err * x_ads).mean()
    b -= lr * err.mean()
    hist.append((w, b))

# Closed-form Normal Equation
wb_normal = np.linalg.lstsq(np.c_[x_ads, np.ones(n)], y_ads, rcond=None)[0]

# 2. Loss surface grid
Wg, Bg = np.meshgrid(np.linspace(-1, 5, 80), np.linspace(-6, 14, 80))
L = (((Wg[..., None] * x_ads + Bg[..., None] - y_ads) ** 2).mean(-1)) / 2.0
hp = np.array(hist)

# Plot 3-panel diagnostics
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.8))

# Panel 1: Fitted Line & Residuals
ax1.scatter(x_ads, y_ads, color="#3b6fb6", s=40, alpha=0.8, label="Data")
x_line = np.linspace(0, 10, 100)
ax1.plot(x_line, w * x_line + b, color="#e45756", lw=2.5, label=f"Fit: y = {w:.2f}x + {b:.2f}")
ax1.vlines(x_ads, y_ads, w * x_ads + b, color="#e45756", alpha=0.35, label="Residuals")
ax1.set_title("Fitted Regression Line & Residuals", fontsize=11, fontweight="bold")
ax1.set_xlabel("Ad Spend (k$)")
ax1.set_ylabel("Units Sold")
ax1.legend(loc="upper left")

# Panel 2: Loss Surface & Gradient Descent Path
ax2.contourf(Wg, Bg, L, levels=np.geomspace(L.min() + 0.5, L.max(), 20), cmap="Blues_r", alpha=0.85)
ax2.plot(hp[::10, 0], hp[::10, 1], color="k", lw=1.2, label="GD Path")
ax2.scatter(w, b, color="#e45756", s=60, zorder=5, label="Final (w, b)")
ax2.scatter(*wb_normal, marker="*", s=160, color="gold", edgecolor="k", zorder=6, label="Normal Eq.")
ax2.set_title("Loss Surface L(w, b) & Descent Path", fontsize=11, fontweight="bold")
ax2.set_xlabel("Slope (w)")
ax2.set_ylabel("Intercept (b)")
ax2.legend()

# Panel 3: Multiple Regression Feature Weights
Zh, _, _ = data.standardize(X_house)
ym = y_house.mean()
wh = np.linalg.lstsq(np.c_[Zh, np.ones(len(y_house))], y_house - ym, rcond=None)[0][:4]
sns.barplot(x=house_feat_names, y=wh, palette="vlag", ax=ax3)
ax3.axhline(0, color="k", linestyle="--", alpha=0.6)
ax3.set_title("Standardized Coefficients (House Prices)", fontsize=11, fontweight="bold")
ax3.set_ylabel("Effect on Price (k$) per 1-SD")

plt.tight_layout()
plt.show()


### Step 4: Industry Benchmark (Comparison with scikit-learn)
We compare our from-scratch parameters against `sklearn.linear_model.LinearRegression` to verify exact analytical and numerical equivalence.


In [ ]:
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression().fit(x_ads.reshape(-1, 1), y_ads)

print("--- Parameter Estimates Comparison ---")
print(f"NumPy Gradient Descent: w = {w:.4f}, b = {b:.4f}")
print(f"scikit-learn OLS:       w = {sk_model.coef_[0]:.4f}, b = {sk_model.intercept_:.4f}")
print(f"Normal Equation:        w = {wb_normal[0]:.4f}, b = {wb_normal[1]:.4f}")
print(f"Mathematical equivalence confirmed within tolerance: {np.isclose(w, sk_model.coef_[0], atol=1e-2)}")


### Step 5: Visualizing Learning Dynamics

Left: the line rotating and shifting toward the data, red segments are the residuals being squared. Right: the loss
surface over (w,b) with the optimizer's path sliding down the elongated valley. Afterwards the script fits a
4-feature *house price* model both ways (gradient descent vs normal equation) and prints the coefficients.


In [ ]:
# Display the learning animation inline
media_file = f"../outputs/03_linear_regression.gif"
if os.path.exists(media_file):
    display(Image(filename=media_file))
elif os.path.exists(f"outputs/03_linear_regression.gif"):
    display(Image(filename=f"outputs/03_linear_regression.gif"))


## Assumptions & pitfalls

- Linear relationship; independent errors with constant variance; no extreme multicollinearity.
- Sensitive to outliers (squared error). Scale features before gradient descent.
- Coefficients are only interpretable when features are not heavily correlated.


## 🧠 Check Your Understanding

<details>
<summary><b>1. Why does squaring the error punish large outliers more than small ones?</b></summary>
Because squaring is a convex function where the derivative $2(y - \hat{y})$ grows linearly with error magnitude. An error of 10 units contributes 100 to the loss, whereas an error of 1 unit contributes only 1.
</details>

<details>
<summary><b>2. When is Gradient Descent preferred over the analytical Normal Equation?</b></summary>
When the number of features $d$ is very large (e.g. $d > 10,000$), calculating the inverse $(X^T X)^{-1}$ requires $O(d^3)$ operations and can exceed memory limits. Gradient Descent scales at $O(k \cdot n \cdot d)$.
</details>


## 🧪 Guided Experiments

- Standardize x before descending: the valley becomes round and convergence is far faster.
- Add one huge outlier to the data and watch the line get dragged toward it.
